In [0]:
dbutils.fs.ls("dbfs:/Volumes/dev/bronze/raw_volume")

#### Creating Widgets

In [0]:
dbutils.widgets.dropdown("env", "dev", ["dev", "uat", "prod"])

In [0]:
env = dbutils.widgets.get("env")

In [0]:
catalog = env          # dev / uat / prod
schema = "bronze"      # fixed
volume = "raw_data"    # fixed

base_path = f"dbfs:/Volumes/{catalog}/{schema}/{volume}/"

In [0]:
from pyspark.sql.functions import current_timestamp, col

#### BRONZE LAYER (RAW INGESTION)

In [0]:
from pyspark.sql.functions import current_timestamp, col

# ─────────────────────────────────────────────
# WIDGET (ONLY ONE)
# ─────────────────────────────────────────────
dbutils.widgets.dropdown("env", "dev", ["dev", "uat", "prod"])

env = dbutils.widgets.get("env")

# ─────────────────────────────────────────────
# DERIVED CONFIG
# ─────────────────────────────────────────────
catalog = env
schema = "bronze"
volume = "raw_volume"

base_path = f"dbfs:/Volumes/{catalog}/{schema}/{volume}/"

# ─────────────────────────────────────────────
# REUSABLE FUNCTION
# ─────────────────────────────────────────────
def load_to_bronze(file_name):
    
    file_path = f"{base_path}{file_name}.csv"
    
    df = spark.read.format("csv") \
        .option("header", True) \
        .load(file_path)
    
    df = df \
        .withColumn("ingestion_timestamp", current_timestamp()) \
        .withColumn("source_file", col("_metadata.file_path"))
    
    table_name = f"{catalog}.{schema}.bronze_{file_name}"
    
    df.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(table_name)
    
    print(f"✅ Loaded {file_name} → {table_name}")

# ─────────────────────────────────────────────
# LOAD ALL FILES
# ─────────────────────────────────────────────
files = ["customers", "products", "orders", "order_items"]

for file in files:
    load_to_bronze(file)